### Uploading and Unzipping a File from your PC

First, you will need to upload the `.zip` file from your local computer to the Colab environment. Once uploaded, you can then unzip its contents.

In [ ]:
from google.colab import files
import os

print("Please upload your zip file(s).")

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

# Get the name of the first uploaded file
zip_file_name = list(uploaded.keys())[0]

Please upload your zip file(s).


Saving archive.osm.zip to archive.osm.zip
User uploaded file "archive.osm.zip" with length 106215656 bytes


In [ ]:
import zipfile

# Check if the uploaded file exists and is a zip file
if os.path.exists(zip_file_name) and zip_file_name.endswith('.zip'):
    print(f"Unzipping '{zip_file_name}'...")
    with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
        zip_ref.extractall('.')  # Extract to the current directory
    print(f"Successfully unzipped '{zip_file_name}'. Contents extracted to the current directory.")
else:
    print(f"Error: '{zip_file_name}' is not a valid zip file or does not exist.")

Unzipping 'archive.osm.zip'...
Successfully unzipped 'archive.osm.zip'. Contents extracted to the current directory.


In [ ]:
print("Listing extracted files and directories:")
for root, dirs, files in os.walk('.'):
    for d in dirs:
        print(os.path.join(root, d) + '/')
    for file in files:
        if file != zip_file_name: # Exclude the original zip file
            print(os.path.join(root, file))

Streaming output truncated to the last 5000 lines.
./Gesture Image Data/A/1209.jpg
./Gesture Image Data/A/1014.jpg
./Gesture Image Data/A/996.jpg
./Gesture Image Data/A/924.jpg
./Gesture Image Data/A/439.jpg
./Gesture Image Data/A/707.jpg
./Gesture Image Data/A/847.jpg
./Gesture Image Data/A/339.jpg
./Gesture Image Data/A/1061.jpg
./Gesture Image Data/A/823.jpg
./Gesture Image Data/A/1323.jpg
./Gesture Image Data/A/121.jpg
./Gesture Image Data/A/154.jpg
./Gesture Image Data/A/712.jpg
./Gesture Image Data/A/625.jpg
./Gesture Image Data/A/774.jpg
./Gesture Image Data/A/1346.jpg
./Gesture Image Data/A/1119.jpg
./Gesture Image Data/A/1413.jpg
./Gesture Image Data/A/729.jpg
./Gesture Image Data/A/594.jpg
./Gesture Image Data/A/563.jpg
./Gesture Image Data/A/367.jpg
./Gesture Image Data/A/714.jpg
./Gesture Image Data/A/87.jpg
./Gesture Image Data/A/258.jpg
./Gesture Image Data/A/857.jpg
./Gesture Image Data/A/887.jpg
./Gesture Image Data/A/1312.jpg
./Gesture Image Data/A/780.jpg
./Gesture Im

In [ ]:
# Install common build tools and Python development headers
!apt-get update -qq
!apt-get install -y -qq build-essential python3-dev cmake

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 118419 files and directories currently installed.)
Preparing to unpack .../00-libpython3.10-dev_3.10.12-1~22.04.17_amd64.deb ...
Unpacking libpython3.10-dev:amd64 (3.10.12-1~22.04.17) over (3.10.12-1~22.04.16) ...
Preparing to unpack .../01-libpython3.10_3.10.12-1~22.04.17_amd64.deb ...
Unpacking libpython3.10:amd64 (3.10.12-1~22.04.17) over (3.10.12-1~22.04.16) ...
Preparing to unpack .../02-python3.10_3.10.12-1~22.04.17_amd64.deb ...
Unpacking python3.10 (3.10.12-1~22.04.17) over (3.10.12-1~22.04.16) ...
Preparing to unpack .../03-libpython3.10-stdlib_3.10.12-1~22.04.17_amd64.deb ...
Unpacking libpython3.10-stdlib:amd64 (3.10.12-1~22.04.17) over (3.10.12-1~22.04.16) ...
Preparing to unpack .../04-python3.10-minimal_3.10.12-1~22.04.17_amd64.deb ...
Unpacking python3.10-minimal 

In [ ]:
!pip install -q mediapipe opencv-python scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 11.5 MB/s eta 0:00:00


In [ ]:
!wget -q -O hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

In [ ]:
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

base_options = mp_python.BaseOptions(model_asset_path='hand_landmarker.task')
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1,              # assumes one hand per sign; change to 2 if any signs use both hands
    running_mode=vision.RunningMode.IMAGE
)
landmarker = vision.HandLandmarker.create_from_options(options)

In [ ]:
import numpy as np

def extract_landmarks(image_path):
    """Takes an image path, returns a 63-number vector (21 landmarks x, y, z) or None if no hand found."""
    image = mp.Image.create_from_file(image_path)
    result = landmarker.detect(image)

    if not result.hand_landmarks:
        return None

    hand = result.hand_landmarks[0]  # first detected hand
    coords = []
    for point in hand:
        coords.extend([point.x, point.y, point.z])

    return np.array(coords)

In [ ]:
import os
from tqdm import tqdm

data_dir = 'Gesture Image Data'  # <-- change to your actual folder path

X, y = [], []
failed_counts = {}

labels = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
print(f"Found {len(labels)} classes: {labels}")

for label in labels:
    label_dir = os.path.join(data_dir, label)
    files = os.listdir(label_dir)
    failed = 0
    for filename in tqdm(files, desc=label):
        img_path = os.path.join(label_dir, filename)
        features = extract_landmarks(img_path)
        if features is not None:
            X.append(features)
            y.append(label)
        else:
            failed += 1
    failed_counts[label] = failed

X = np.array(X)
y = np.array(y)

print(f"\nLoaded {len(X)} total samples")
print("\nImages with no hand detected, per class:")
for label, count in failed_counts.items():
    print(f"  {label}: {count}")

Found 37 classes: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '_']


_: 100%|██████████| 1500/1500 [01:03<00:00, 23.53it/s]


Loaded 30249 total samples

Images with no hand detected, per class:
  0: 1452
  1: 1368
  2: 1392
  3: 1483
  4: 621
  5: 1121
  6: 1440
  7: 1442
  8: 1321
  9: 1424
  A: 737
  B: 0
  C: 100
  D: 1
  E: 1459
  F: 0
  G: 21
  H: 0
  I: 217
  J: 541
  K: 209
  L: 0
  M: 1490
  N: 474
  O: 1485
  P: 14
  Q: 36
  R: 5
  S: 1454
  T: 1186
  U: 13
  V: 8
  W: 24
  X: 3
  Y: 1387
  Z: 407
  _: 916


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)

# First split off the test set (never touched until the very end)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y_encoded, test_size=0.15, random_state=42, stratify=y_encoded
)

# Then split the remainder into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp
)
# 0.1765 of the remaining 85% ≈ 15% of the original total, so overall ~70/15/15

print(f"Train: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Val:   {len(X_val)} ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test:  {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

Train: 21173 (70.0%)
Val:   4538 (15.0%)
Test:  4538 (15.0%)


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

clf = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300, random_state=42)
clf.fit(X_train, y_train)

# Check performance on validation set while experimenting
val_pred = clf.predict(X_val)
print("=== Validation performance ===")
print(classification_report(y_val, val_pred, target_names=le.classes_))

=== Validation performance ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         7
           1       0.83      1.00      0.91        20
           2       0.94      1.00      0.97        16
           3       1.00      1.00      1.00         2
           4       1.00      1.00      1.00       132
           5       0.98      0.98      0.98        57
           6       1.00      0.89      0.94         9
           7       0.82      1.00      0.90         9
           8       1.00      0.93      0.96        27
           9       1.00      0.91      0.95        11
           A       1.00      0.98      0.99       115
           B       1.00      1.00      1.00       225
           C       1.00      0.99      0.99       210
           D       1.00      1.00      1.00       225
           E       1.00      0.50      0.67         6
           F       1.00      1.00      1.00       225
           G       1.00      1.00      1.00       

In [ ]:
import joblib

joblib.dump(clf, 'sign_classifier.pkl')
joblib.dump(le, 'label_encoder.pkl')  # needed to convert predictions back into letters

['label_encoder.pkl']